# Fine-Tune Qwen2.5-7B for Arabic Teaching (SageMaker)

**Model:** Qwen2.5-7B-Instruct with LoRA (using Unsloth)

**Training data:** ~115 conversations (teaching, grading, exercise generation)

**Method:** LoRA (rank=32, alpha=32)

**Max sequence length:** 1536 tokens

**Output:** 4-bit quantized model (~2GB)

---

## Setup for SageMaker

1. **Instance Type:** ml.g4dn.xlarge or ml.g5.xlarge (T4/A10G GPU)
2. **HF Token:** Set as environment variable `HF_TOKEN` or in cell below
3. **Training data:** Upload `combined_training_data.jsonl` to EFS at `/home/sagemaker-user/user-default-efs/arabic-teaching/data/`
4. **Run all cells** (~20-30 minutes)
5. **Model output:** Saved to EFS at `/home/sagemaker-user/user-default-efs/arabic-teaching/models/qwen-7b-arabic-teaching/`

## Cell 1: Install Dependencies

In [1]:
%%capture
# Simple installation - use SageMaker's default PyTorch
!pip install --upgrade torchvision
!pip install unsloth trl transformers datasets accelerate bitsandbytes
!pip install -U peft
!pip uninstall -y pyarrow
!pip install pyarrow --no-cache-dir

## ⚠️ RESTART KERNEL - Then run all cells below

## Cell 2: HuggingFace Auth

In [ ]:
# Check what's using space
!du -sh /home/sagemaker-user/* 2>/dev/null | sort -h                                                                                                                            

# Clear everything possible from default storage
!rm -rf ~/.cache/*                                                                                                                                                              
!rm -rf /home/sagemaker-user/.cache/*                                                                                                                                           


import os
from huggingface_hub import login

EFS_ROOT = "/home/sagemaker-user/user-default-efs"
HF_CACHE = os.path.join(EFS_ROOT, "hf_cache")

# 1. Create the directories on the BIG drive
os.makedirs(HF_CACHE, exist_ok=True)

# 2. Redirect ALL possible cache/temp variables
os.environ["HF_HOME"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = os.path.join(EFS_ROOT, "hf_cache")
os.environ["HF_DATASETS_CACHE"] = os.path.join(EFS_ROOT, "hf_cache")

# Now check space
!df -h /home/sagemaker-user      

print(f"✅ All traffic redirected to EFS: {HF_CACHE}")
login(token=<HF_TOKEN>)

0	/home/sagemaker-user/user-default-efs
Filesystem      Size  Used Avail Use% Mounted on
/dev/nvme2n1    5.0G   70M  4.9G   2% /home/sagemaker-user
✅ All traffic redirected to EFS: /home/sagemaker-user/user-default-efs/hf_cache


## Cell 3: Imports & Check GPU

In [3]:
import gc
import json
import os

import torch

# IMPORTANT: Disable Unsloth statistics BEFORE importing Unsloth
os.environ["UNSLOTH_DISABLE_LOG_STATS"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
if torch.cuda.is_available():
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected! Training will be very slow.")

/opt/conda/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-04-16 03:55:12.793048: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776311712.814930    5175 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776311712.822810    5175 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-16 03:55:12.859625: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SS

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch: 2.10.0+cu128
GPU: NVIDIA L4
Memory: 23.7 GB


## Cell 4: Configuration

In [ ]:
# EFS paths (adjust if your EFS mount is different)
EFS_ROOT = "/home/sagemaker-user/user-default-efs"
DATA_DIR = f"{EFS_ROOT}/data"

# Combined output
COMBINED_DATA = f"{DATA_DIR}/agent2_grading_training_data.jsonl"

print("📦 Combining training data files...\n")
DATA_PATH = f"{COMBINED_DATA}"
OUTPUT_DIR = f"{EFS_ROOT}/models/qwen-7b-arabic-teaching"

# Use the non-Unsloth version (Unsloth will patch it automatically)
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LENGTH = 512
LOAD_IN_4BIT = True

# LoRA config
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0
LORA_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training config
BATCH_SIZE = 4
GRAD_ACCUM = 4
EPOCHS = 5
LR = 2e-4
WARMUP = 0.03

print("=" * 60)
print("🚀 Fine-Tuning Qwen2.5-7B for Arabic Teaching")
print("=" * 60)
print(f"Model: {MODEL_NAME}")
print(f"Data: {DATA_PATH}")
print(f"Output: {OUTPUT_DIR}")
print(f"Batch: {BATCH_SIZE}, Accum: {GRAD_ACCUM}, Epochs: {EPOCHS}")
print(f"Max Seq Length: {MAX_SEQ_LENGTH}")
print("=" * 60)

# Verify data file exists
if not os.path.exists(DATA_PATH):
    print(f"\n❌ Training data not found at: {DATA_PATH}")
    print("\nPlease upload combined_training_data.jsonl to EFS at:")
    print(f"   {os.path.dirname(DATA_PATH)}/")
else:
    print("\n✅ Training data found")

# Verify EFS cache is setup
print(f"\n📁 HuggingFace cache: {os.environ.get('HF_HOME', 'NOT SET')}")
if os.path.exists(os.environ.get("HF_HOME", "")):
    print("✅ EFS cache directory exists")
else:
    print("❌ EFS cache directory NOT found - run Cell 2 first!")

📦 Combining training data files...

🚀 Fine-Tuning Qwen2.5-7B for Arabic Teaching
Model: Qwen/Qwen2.5-7B-Instruct
Data: /home/sagemaker-user/user-default-efs/data/agent2_grading_training_data.jsonl
Output: /home/sagemaker-user/user-default-efs/models/qwen-7b-arabic-teaching
Batch: 4, Accum: 4, Epochs: 5
Max Seq Length: 512

✅ Training data found

📁 HuggingFace cache: /home/sagemaker-user/user-default-efs/hf_cache
✅ EFS cache directory exists


## Cell 5: Load Training Data

In [5]:
conversations = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            conversations.append(json.loads(line))

print(f"✅ Loaded {len(conversations)} conversations")
print(f"   Steps/epoch: {len(conversations) // (BATCH_SIZE * GRAD_ACCUM)}")
print(f"   Total steps: {(len(conversations) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS}")

✅ Loaded 345 conversations
   Steps/epoch: 21
   Total steps: 105


## Cell 6: Load Model with Unsloth

In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
    token="<HF_TOKEN>",
)
print(f"✅ Model loaded: {model.num_parameters() / 1e9:.2f}B params")

==((====))==  Unsloth 2026.4.5: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Model loaded: 7.62B params


## Cell 7: Add LoRA Adapters

## Cell 8: Format Dataset

In [7]:
formatted_data = [
    {
        "text": tokenizer.apply_chat_template(
            c["messages"], tokenize=False, add_generation_prompt=False
        )
    }
    for c in conversations
]

dataset = Dataset.from_list(formatted_data)
print(f"✅ Dataset: {len(dataset)} examples")


✅ Dataset: 345 examples


In [8]:
# Validation split                                                                                                                
eval_size = 20  # Use 20 examples for validation                                                                                  
train_dataset = dataset.select(range(len(dataset) - eval_size))                                                                   
eval_dataset = dataset.select(range(len(dataset) - eval_size, len(dataset)))                                                      
                                                                                                                                
print(f"✅ Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")                                                               
                                                                       

✅ Train: 325, Eval: 20


## Cell 9: Setup Trainer

In [9]:
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
import torch
import gc

# 1. Clear memory
torch.cuda.empty_cache()
gc.collect()

# 2. Define the PEFT configuration (matches your LORA settings)
peft_config = LoraConfig(
    r = LORA_R,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    target_modules = LORA_MODULES,
    bias = "none",
    task_type = "CAUSAL_LM",
)

model.add_adapter(peft_config, adapter_name="my_adapter")

# 3. Define the SFTConfig (v1.0.0 requirement)
sft_config = SFTConfig(
    output_dir = OUTPUT_DIR,
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_text_field = "text",
    packing = False,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate = LR,
    num_train_epochs = EPOCHS,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    optim = "adamw_8bit",
    seed = 3407,
    eval_strategy="epoch",  # ADD THIS                                                                                      
    save_strategy="epoch",  # ADD THIS                                                                                            
    load_best_model_at_end=True,  # ADD THIS                                                                                      
    metric_for_best_model="loss",  # ADD THIS                                                                                                                             
)

# 4. Initialize Trainer exactly as shown in the second docs
trainer = SFTTrainer(
    model = model,
    train_dataset = train_dataset,
    eval_dataset=eval_dataset,
    args = sft_config,
    processing_class = tokenizer,  # <--- Replaces 'tokenizer'
)

trainer.model.set_adapter("my_adapter")

print("✅ Trainer ready! The PEFT config is now explicitly linked.")

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/325 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/20 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
✅ Trainer ready! The PEFT config is now explicitly linked.


## Cell 10: Train (~20-30 min on g4dn.xlarge)

In [10]:
import logging

# Disable the specific logger that's crashing on the closed Jupyter stream
logging.getLogger("transformers.trainer").setLevel(logging.ERROR)

print("🚀 Training starting...")

try:
    trainer.train()
except ValueError as e:
    if "I/O operation on closed file" in str(e):
        print("Caught Jupyter logging error, but training is proceeding in the background.")
        # Training often continues even if the progress bar display fails
    else:
        raise e

print("\n✅ Check your EFS 'models' folder for checkpoints!")

🚀 Training starting...
Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,0.515400,1.518199
2,0.138600,0.602573
3,0.126800,0.566584
4,0.118300,0.586316
5,0.087700,0.599725


FileNotFoundError: [Errno 2] No such file or directory: '/home/sagemaker-user/user-default-efs/models/qwen-7b-arabic-teaching/checkpoint-63/pytorch_model.bin'

## Cell 11: Save Model to EFS

In [ ]:
# Cell 11: Save Model to EFS
import os
import tarfile

# Save to EFS (persists across notebook sessions)
lora_output_dir = f"{OUTPUT_DIR}/lora_adapters"
model.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)

print(f"✅ Saved to EFS: {lora_output_dir}")
print("\n📁 Model persisted on EFS at:")
print(f"   {lora_output_dir}")

print("\n📦 Creating downloadable archive...")

# Create tarball in /tmp (accessible for download)
archive_path = f"{OUTPUT_DIR}/qwen-7b-arabic-teaching-lora.tar.gz"
with tarfile.open(archive_path, "w:gz") as tar:
    tar.add(lora_output_dir, arcname="lora_adapters")

archive_size = os.path.getsize(archive_path) / (1024 * 1024)
print(f"✅ Archive created: {archive_path} ({archive_size:.1f} MB)")

print("\n📥 To download:")
print("   1. File browser → /tmp/qwen-7b-arabic-teaching-lora.tar.gz")
print("   2. Right-click → Download")
print("   3. Extract on your machine")

print("\nModel files in archive:")
for file in os.listdir(lora_output_dir):
    file_path = os.path.join(lora_output_dir, file)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"   - {file} ({size_mb:.1f} MB)")

✅ Saved to EFS: /home/sagemaker-user/user-default-efs/models/qwen-7b-arabic-teaching/lora_adapters

📁 Model persisted on EFS at:
   /home/sagemaker-user/user-default-efs/models/qwen-7b-arabic-teaching/lora_adapters

📦 Creating downloadable archive...
✅ Archive created: /home/sagemaker-user/user-default-efs/models/qwen-7b-arabic-teaching/qwen-7b-arabic-teaching-lora.tar.gz (18.8 MB)

📥 To download:
   1. File browser → /tmp/qwen-3b-arabic-teaching-lora.tar.gz
   2. Right-click → Download
   3. Extract on your machine

Model files in archive:
   - tokenizer_config.json (0.0 MB)
   - adapter_model.safetensors (19.3 MB)
   - adapter_config.json (0.0 MB)
   - merges.txt (1.6 MB)
   - generation_config.json (0.0 MB)
   - special_tokens_map.json (0.0 MB)
   - chat_template.jinja (0.0 MB)
   - added_tokens.json (0.0 MB)
   - tokenizer.json (10.9 MB)
   - vocab.json (2.6 MB)


## Cell 12: Test Generation

In [12]:
# Prepare for inference
FastLanguageModel.for_inference(model)

print("🧪 Testing JSON format compliance (PRIMARY GOAL)...\n")
print("Expected: ONLY {\"correct\": true} or {\"correct\": false}")
print("No extra text, no explanations\n")
print("=" * 60)

# Test 1: Correct vocabulary answer
test_messages = [
    {"role": "system", "content": "You are an Arabic vocabulary grading system. Return only valid JSON."},
    {
        "role": "user",
        "content": """Mode: grading_vocab

Question: What does \"كِتَابٌ\" mean?
Student Answer: \"book\"
Correct Answer: \"book\"

Evaluate if the student's answer is correct.
Return JSON:
{\"correct\": true} or {\"correct\": false}

Response:""",
    },
]

prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=50, temperature=0.1, do_sample=False)
full_output = tokenizer.batch_decode(outputs)[0]

# Extract assistant response
if "<|im_start|>assistant" in full_output:
    response = full_output.split("<|im_start|>assistant")[-1]
    response = response.replace("<|im_end|>", "").strip()
else:
    response = full_output

print("Test 1: Correct answer (should be {\"correct\": true})")
print(f"Response: {response}")
print("=" * 60)

# Test 2: Incorrect vocabulary answer
test_messages_2 = [
    {"role": "system", "content": "You are an Arabic vocabulary grading system. Return only valid JSON."},
    {
        "role": "user",
        "content": """Mode: grading_vocab

Question: What does \"مَدْرَسَةٌ\" mean?
Student Answer: \"house\"
Correct Answer: \"school\"

Evaluate if the student's answer is correct.
Return JSON:
{\"correct\": true} or {\"correct\": false}

Response:""",
    },
]

prompt_2 = tokenizer.apply_chat_template(test_messages_2, tokenize=False, add_generation_prompt=True)
inputs_2 = tokenizer([prompt_2], return_tensors="pt").to("cuda")

outputs_2 = model.generate(**inputs_2, max_new_tokens=50, temperature=0.1, do_sample=False)
full_output_2 = tokenizer.batch_decode(outputs_2)[0]

if "<|im_start|>assistant" in full_output_2:
    response_2 = full_output_2.split("<|im_start|>assistant")[-1]
    response_2 = response_2.replace("<|im_end|>", "").strip()
else:
    response_2 = full_output_2

print("\nTest 2: Incorrect answer (should be {\"correct\": false})")
print(f"Response: {response_2}")
print("=" * 60)

# Validate JSON format
print("\n🔍 Validation:")
try:
    json.loads(response)
    print("✅ Test 1: Valid JSON")
except:
    print("❌ Test 1: INVALID JSON - needs more training or post-processing")

try:
    json.loads(response_2)
    print("✅ Test 2: Valid JSON")
except:
    print("❌ Test 2: INVALID JSON - needs more training or post-processing")

🧪 Testing JSON format compliance (PRIMARY GOAL)...

Expected: ONLY {"correct": true} or {"correct": false}
No extra text, no explanations

Test 1: Correct answer (should be {"correct": true})
Response: {"correct": true}

Test 2: Incorrect answer (should be {"correct": false})
Response: {"correct": false}

🔍 Validation:
✅ Test 1: Valid JSON
✅ Test 2: Valid JSON


In [13]:
print("🧪 Testing harakaat flexibility (SECONDARY GOAL)...\n")
print("Expected: Accept variations in internal harakaat")
print("Required: Case endings must match\n")
print("=" * 60)

# Test: Student answer without internal harakaat, but correct case ending
test_harakaat = [
    {"role": "system", "content": "You are an Arabic vocabulary grading system. Return only valid JSON."},
    {
        "role": "user",
        "content": """Mode: grading_vocab

Question: Translate: the book
Student Answer: \"الكتابُ\"
Correct Answer: \"الكِتَابُ\"

Evaluate if the student's answer is correct. Note: Accept answers with or without harakaat.
Return JSON:
{\"correct\": true} or {\"correct\": false}

Response:""",
    },
]

prompt_h = tokenizer.apply_chat_template(test_harakaat, tokenize=False, add_generation_prompt=True)
inputs_h = tokenizer([prompt_h], return_tensors="pt").to("cuda")

outputs_h = model.generate(**inputs_h, max_new_tokens=50, temperature=0.1, do_sample=False)
full_output_h = tokenizer.batch_decode(outputs_h)[0]

if "<|im_start|>assistant" in full_output_h:
    response_h = full_output_h.split("<|im_start|>assistant")[-1]
    response_h = response_h.replace("<|im_end|>", "").strip()
else:
    response_h = full_output_h

print("Test: الكتابُ vs الكِتَابُ (missing internal harakaat, has case ending)")
print(f"Response: {response_h}")
print("Expected: {\"correct\": true}")
print("=" * 60)

# Validate
try:
    result = json.loads(response_h)
    if result.get("correct") == True:
        print("\n✅ Harakaat flexibility: WORKING")
    else:
        print("\n⚠️  Harakaat flexibility: Model rejected (may need more training)")
except:
    print("\n❌ Invalid JSON response")

🧪 Testing harakaat flexibility (SECONDARY GOAL)...

Expected: Accept variations in internal harakaat
Required: Case endings must match

Test: الكتابُ vs الكِتَابُ (missing internal harakaat, has case ending)
Response: {"correct": true}
Expected: {"correct": true}

✅ Harakaat flexibility: WORKING


In [14]:
print("🧪 Testing synonym acceptance...\n")
print("=" * 60)

# Test: Synonym (instructor = teacher)
test_synonym = [
    {"role": "system", "content": "You are an Arabic vocabulary grading system. Return only valid JSON."},
    {
        "role": "user",
        "content": """Mode: grading_vocab

Question: What does \"مُعَلِّمٌ\" mean?
Student Answer: \"instructor\"
Correct Answer: \"teacher\"

Evaluate if the student's answer is correct. Be flexible: accept synonyms.
Return JSON:
{\"correct\": true} or {\"correct\": false}

Response:""",
    },
]

prompt_s = tokenizer.apply_chat_template(test_synonym, tokenize=False, add_generation_prompt=True)
inputs_s = tokenizer([prompt_s], return_tensors="pt").to("cuda")

outputs_s = model.generate(**inputs_s, max_new_tokens=50, temperature=0.1, do_sample=False)
full_output_s = tokenizer.batch_decode(outputs_s)[0]

if "<|im_start|>assistant" in full_output_s:
    response_s = full_output_s.split("<|im_start|>assistant")[-1]
    response_s = response_s.replace("<|im_end|>", "").strip()
else:
    response_s = full_output_s

print("Test: instructor = teacher (synonym)")
print(f"Response: {response_s}")
print("Expected: {\"correct\": true}")
print("=" * 60)

# Validate
try:
    result = json.loads(response_s)
    if result.get("correct") == True:
        print("\n✅ Synonym acceptance: WORKING")
    else:
        print("\n⚠️  Synonym acceptance: Model rejected (may need more training)")
except:
    print("\n❌ Invalid JSON response")

🧪 Testing synonym acceptance...

Test: instructor = teacher (synonym)
Response: {"correct": true}
Expected: {"correct": true}

✅ Synonym acceptance: WORKING


In [15]:
# Cell: Evaluation Dataset for Training Validation                                                                                
                                                                                                                                    
import json                                                                                                                       
                                                                                                                                
GRADING_EVAL_CASES = [                                                                                                            
  # Test 1: Major typo (2+ characters missing) - should REJECT
  {                                                                                                                             
      "messages": [                   
          {"role": "system", "content": "You are an Arabic vocabulary grading system. Return only valid JSON."},                
          {"role": "user", "content": """Mode: grading_vocab                                                                    
                                                                                                                                
Arabic Word: ملَقَ (qalam)                                                                                                          
Correct Answer: pen                                                                                                               
Student Answer: pn                                                                                                                
                                                                                                                                
Evaluate if the student answer is correct. Consider:                                                                              
- Minor typos (1 letter) are acceptable                                                                                           
- Major typos (2+ letters) should be rejected                                                                                     
- Must be the exact vocabulary item, not related words                                                                            
                                                                                                                                
Return JSON:                                                                                                                      
{"correct": true} or {"correct": false}                                                                                           
                                                                                                                                
Response:"""}
      ],                                                                                                                        
      "expected": {"correct": False}, 
      "explanation": "Missing 33% of letters (e). This is a major typo."                                                        
  },                                                                                                                            
                                                                                                                                
  # Test 2: Wrong vocabulary (semantically related) - should REJECT                                                             
  {                                   
      "messages": [                                                                                                             
          {"role": "system", "content": "You are an Arabic vocabulary grading system. Return only valid JSON."},                
          {"role": "user", "content": """Mode: grading_vocab                                                                    
                                                                                                                                
Arabic Word: ملَقَ (qalam)                                                                                                          
Correct Answer: pen                                                                                                               
Student Answer: pencil                  
                                                                                                                                
Evaluate if the student answer is correct. Consider:
- Exact vocabulary match required                                                                                                 
- Pen (ملَقَ) uses ink                                                                                                              
- Pencil (صاصَرَ ملَقَ) uses graphite                                                                                                 
- These are DIFFERENT objects, not synonyms                                                                                       
                                                                                                                                
Return JSON:                                                                                                                      
{"correct": true} or {"correct": false}                                                                                           
                                                                                                                                
Response:"""}                                                                                                                     
      ],                                                                                                                        
      "expected": {"correct": False}, 
      "explanation": "Pencil is not pen. ملَقَ means pen (ink), not pencil (صاصَرَ ملَقَ)."                                           
  },                                                                                                                            
                                                                                                                                
  # Test 3: Harakaat variations - should ACCEPT                                                                                 
  {                                                                                                                             
      "messages": [                                                                                                             
          {"role": "system", "content": "You are an Arabic vocabulary grading system. Return only valid JSON."},                
          {"role": "user", "content": """Mode: grading_vocab                                                                    
                                                                                                                                
Arabic Word: باتك (without harakaat)                                                                                              
Correct Answer: book                                                                                                              
Student Answer: book                                                                                                              
                                                                                                                                
Note: Student saw باتك but correct answer is for باتَكِ.                                                                            
Harakaat variations are acceptable.                                                                                               
                                                                                                                                
Return JSON:                                                                                                                      
{"correct": true} or {"correct": false}                                                                                           
                                                                                                                                
Response:"""}
      ],                                                                                                                        
      "expected": {"correct": True},                                                                                            
      "explanation": "Correct. Harakaat variations are acceptable."                                                             
  },                                                                                                                            
                                                                                                                                
  # Test 4: Case variations - should ACCEPT                                                                                     
  {                                   
      "messages": [                                                                                                             
          {"role": "system", "content": "You are an Arabic vocabulary grading system. Return only valid JSON."},
          {"role": "user", "content": """Mode: grading_vocab                                                                    
                                                                                                                                
Arabic Word: ةسَرَدْمَ (madrasa)                                                                                                      
Correct Answer: school                                                                                                            
Student Answer: School                                                                                                            
                                                                                                                                
Case differences should be ignored.                                                                                               
                                                                                                                                
Return JSON:                            
{"correct": true} or {"correct": false}                                                                                           
                                                                                                                                
Response:"""}
      ],                                                                                                                        
      "expected": {"correct": True},                                                                                            
      "explanation": "Correct. Case differences are acceptable."                                                                
  },                                                                                                                            
                                                                                                                                
  # Test 5: Single letter typo - should ACCEPT                                                                                  
  {                                   
      "messages": [                                                                                                             
          {"role": "system", "content": "You are an Arabic vocabulary grading system. Return only valid JSON."},
          {"role": "user", "content": """Mode: grading_vocab                                                                    
                                                                                                                                
Arabic Word: ةسَرَدْمَ (madrasa)                                                                                                      
Correct Answer: school                                                                                                            
Student Answer: scool                                                                                                             
                                                                                                                                
Single letter typo - should accept with minor correction.                                                                         
                                                                                                                                
Return JSON:                            
{"correct": true} or {"correct": false}                                                                                           
                                                                                                                                
Response:"""}
      ],                                                                                                                        
      "expected": {"correct": True},  
      "explanation": "Correct despite minor typo (missing h). Student clearly knows the word."                                  
  },                                                                                                                            
                                                                                                                                
  # Test 6: Completely wrong - should REJECT                                                                                    
  {                                   
      "messages": [                                                                                                             
          {"role": "system", "content": "You are an Arabic vocabulary grading system. Return only valid JSON."},
          {"role": "user", "content": """Mode: grading_vocab                                                                    
                                                                                                                                
Arabic Word: باتَكِ (kitaab)                                                                                                        
Correct Answer: book                                                                                                              
Student Answer: school                                                                                                            
                                                                                                                                
Completely different word.                                                                                                        
                                                                                                                                
Return JSON:                            
{"correct": true} or {"correct": false}                                                                                           
                                                                                                                                
Response:"""}
      ],                                                                                                                        
      "expected": {"correct": False},                                                                                           
      "explanation": "Incorrect. باتَكِ means book, not school."                                                                  
  },                                                                                                                            
]                                                                                                                                 
                                                                                                                                
def evaluate_model_on_test_cases(model, tokenizer, test_cases):
  """                                                                                                                           
  Run evaluation on test cases during/after training.

  Returns accuracy metrics and detailed results.
  """
  results = []
  correct_count = 0

  FastLanguageModel.for_inference(model)

  for i, test in enumerate(test_cases):
      # Format prompt
      prompt = tokenizer.apply_chat_template(
          test["messages"],
          tokenize=False,
          add_generation_prompt=True                                                                                            
      )                                                                                                                         
                                                                                                                                
      # Generate response                                                                                                       
      inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
      outputs = model.generate(
          **inputs,
          max_new_tokens=50,
          temperature=0.1,
          do_sample=False
      )
      full_output = tokenizer.batch_decode(outputs)[0]

      # Extract assistant response
      if "<|im_start|>assistant" in full_output:
          response = full_output.split("<|im_start|>assistant")[-1]
          response = response.replace("<|im_end|>", "").strip()
      else:
          response = full_output

      # Parse JSON
      try:
          parsed = json.loads(response)
          is_correct = parsed.get("correct") == test["expected"]["correct"]
          if is_correct:                                                                                                        
              correct_count += 1
                                                                                                                                
          results.append({            
              "test_num": i + 1,
              "expected": test["expected"]["correct"],                                                                          
              "got": parsed.get("correct"),
              "passed": is_correct,                                                                                             
              "response": response,   
              "explanation": test["explanation"]
          })
      except json.JSONDecodeError:
          results.append({
              "test_num": i + 1,
              "expected": test["expected"]["correct"],
              "got": "INVALID_JSON",
              "passed": False,
              "response": response,
              "explanation": test["explanation"]
          })

  accuracy = correct_count / len(test_cases)                                                                                    

  return {                                                                                                                      
      "accuracy": accuracy,           
      "correct": correct_count,
      "total": len(test_cases),
      "results": results
  }

# Run evaluation after training (add this after Cell 11: Train)
print("\n" + "=" * 60)
print("🧪 EVALUATION ON EDGE CASES")                                                                                              
print("=" * 60)
                                                                                                                                
eval_results = evaluate_model_on_test_cases(model, tokenizer, GRADING_EVAL_CASES)

print(f"\n📊 Overall Accuracy: {eval_results['accuracy']:.1%} ({eval_results['correct']}/{eval_results['total']})")               
                                          
# Detailed breakdown                                                                                                              
print("\n📋 Detailed Results:")         
print("-" * 60)                                                                                                                   
                                                                                                                                
for result in eval_results["results"]:                                                                                            
  status = "✅ PASS" if result["passed"] else "❌ FAIL"                                                                         
  print(f"\n{status} - Test {result['test_num']}")                                                                              
  print(f"   Expected: {result['expected']}")                                                                                   
  print(f"   Got: {result['got']}")                                                                                             
  print(f"   Context: {result['explanation']}")                                                                                 
                                                                                                                                
  if not result["passed"]:                                                                                                      
      print(f"   Model Response: {result['response'][:100]}...")                                                                
                                                                                                                                
# Specific failures                                                                                                               
print("\n" + "=" * 60)                                                                                                            
print("🎯 TARGET IMPROVEMENTS:")                                                                                                  
print("=" * 60)                                                                                                                   
                                                                                                                                
test_2_result = eval_results['results'][0]  # Major typo test                                                                     
test_3_result = eval_results['results'][1]  # Wrong vocab test                                                                    
                                                                                                                                
if not test_2_result['passed']:                                                                                                   
  print("❌ Test 2 (Major typo): Still accepting 'pn' for 'pen'")                                                               
  print("   Need: stronger penalty for 33%+ character difference")                                                              
else:                                                                                                                             
  print("✅ Test 2 (Major typo): FIXED - now correctly rejects 'pn'")                                                           
                                                                                                                                
if not test_3_result['passed']:                                                                                                   
  print("❌ Test 3 (Wrong vocab): Still accepting 'pencil' for 'pen'")                                                          
  print("   Need: exact vocabulary matching, not semantic similarity")                                                          
else:                                                                                                                             
  print("✅ Test 3 (Wrong vocab): FIXED - now correctly rejects 'pencil'")                                                      
                                                                                                                                
print("\n" + "=" * 60) 


🧪 EVALUATION ON EDGE CASES

📊 Overall Accuracy: 100.0% (6/6)

📋 Detailed Results:
------------------------------------------------------------

✅ PASS - Test 1
   Expected: False
   Got: False
   Context: Missing 33% of letters (e). This is a major typo.

✅ PASS - Test 2
   Expected: False
   Got: False
   Context: Pencil is not pen. ملَقَ means pen (ink), not pencil (صاصَرَ ملَقَ).

✅ PASS - Test 3
   Expected: True
   Got: True
   Context: Correct. Harakaat variations are acceptable.

✅ PASS - Test 4
   Expected: True
   Got: True
   Context: Correct. Case differences are acceptable.

✅ PASS - Test 5
   Expected: True
   Got: True
   Context: Correct despite minor typo (missing h). Student clearly knows the word.

✅ PASS - Test 6
   Expected: False
   Got: False
   Context: Incorrect. باتَكِ means book, not school.

🎯 TARGET IMPROVEMENTS:
✅ Test 2 (Major typo): FIXED - now correctly rejects 'pn'
✅ Test 3 (Wrong vocab): FIXED - now correctly rejects 'pencil'

